In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import average_precision_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, Model

CSV_PATH = "/content/drive/MyDrive/iot/train_test_network (2).csv"  # <<< 改
df = pd.read_csv(CSV_PATH)

In [22]:
assert "name" in df.columns and "Group" in df.columns and "week" in df.columns
df["name"] = df["name"].astype(str)
df["Group"] = pd.to_numeric(df["Group"], errors="coerce").astype("Int64")
df["week"]  = pd.to_numeric(df["week"],  errors="coerce").astype("Int64")
df = df.dropna(subset=["Group","week"]).copy()
df["Group"] = df["Group"].astype(int)
df["week"]  = df["week"].astype(int)

In [23]:
df["mode"] = np.where(df["week"].isin([1,2]), "active",
               np.where(df["week"].isin([3,4]), "idle", "unknown"))
df = df[df["mode"] != "unknown"].copy()

print("Mode counts:", df["mode"].value_counts().to_dict())


Mode counts: {'active': 133649, 'idle': 77394}


In [24]:
time_candidates = ["ts","time","start_time","timestamp","frame.time_epoch"]
time_col = next((c for c in time_candidates if c in df.columns), None)
if time_col is None:
    df["_row_order"] = np.arange(len(df))
    time_col = "_row_order"
df[time_col] = pd.to_numeric(df[time_col], errors="coerce").fillna(0.0)
df = df.sort_values(["Group", time_col]).reset_index(drop=True)

In [25]:
num_candidates = [
    "duration","src_bytes","dst_bytes","src_pkts","dst_pkts"
    
]
cat_candidates = [
    "proto","service","conn_state",
    "dns_AA","dns_RD","dns_RA","dns_rejected",
    "ssl_version","ssl_cipher","ssl_resumed","ssl_established"
]

num_cols = [c for c in num_candidates if c in df.columns]
cat_cols = []

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0.0)

for pcol in ["src_port","dst_port"]:
    if pcol in df.columns:
        df[pcol] = pd.to_numeric(df[pcol], errors="coerce").fillna(-1).astype(int)
    else:
        df[pcol] = -1

def port_bucket(p):
    if p == 53: return "p53_dns"
    if p == 80: return "p80_http"
    if p == 443: return "p443_https"
    if p == 123: return "p123_ntp"
    if p == 1883: return "p1883_mqtt"
    if p == 5683: return "p5683_coap"
    if p == -1: return "pNA"
    if p < 1024: return "pWellKnownOther"
    return "pHigh"

df["dst_port_bucket"] = df["dst_port"].map(port_bucket)
df["src_port_bucket"] = df["src_port"].map(port_bucket)
cat_cols += ["dst_port_bucket","src_port_bucket"]

for c in cat_cols:
    df[c] = df[c].astype(str).fillna("-")

In [26]:
le = LabelEncoder()
df["y"] = le.fit_transform(df["Group"].values)
n_cls = len(le.classes_)
print("13-way classes:", n_cls, "groups:", list(le.classes_))

# build train/test indices by time within each group
train_frac = 0.7
train_idx = []
test_idx  = []
for g, idxs in df.groupby("Group").indices.items():
    idxs = np.array(idxs)
    k = max(1, int(len(idxs) * train_frac))
    train_idx.extend(idxs[:k])
    test_idx.extend(idxs[k:])

train_idx = np.array(train_idx)
test_idx  = np.array(test_idx)

df_train = df.loc[train_idx].copy()
df_test  = df.loc[test_idx].copy()

print("Train mode counts:", df_train["mode"].value_counts().to_dict())
print("Test  mode counts:", df_test["mode"].value_counts().to_dict())


13-way classes: 13 groups: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]
Train mode counts: {'active': 93280, 'idle': 54445}
Test  mode counts: {'active': 40369, 'idle': 22949}


In [31]:
train_frac = 0.3
train_idx = []
test_idx  = []
for g, idxs in df.groupby("Group").indices.items():
    idxs = np.array(idxs)
    k = max(1, int(len(idxs) * train_frac))
    train_idx.extend(idxs[:k])   # 早期
    test_idx.extend(idxs[k:])    # 晚期

train_idx = np.array(train_idx)
test_idx  = np.array(test_idx)

df_train = df.loc[train_idx].copy()
df_test  = df.loc[test_idx].copy()

print("Train mode counts:", df_train["mode"].value_counts().to_dict())
print("Test  mode counts:", df_test["mode"].value_counts().to_dict())

# preprocess fit on train only
scaler = StandardScaler()
Xn_tr = scaler.fit_transform(df_train[num_cols].values)
Xn_te = scaler.transform(df_test[num_cols].values)

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
Xc_tr = ohe.fit_transform(df_train[cat_cols])
Xc_te = ohe.transform(df_test[cat_cols])

X_tr = np.hstack([Xn_tr, Xc_tr]).astype(np.float32)
X_te = np.hstack([Xn_te, Xc_te]).astype(np.float32)

y_tr = df_train["y"].values
y_te = df_test["y"].values
g_tr = df_train["Group"].values
g_te = df_test["Group"].values
mode_te = df_test["mode"].values



Train mode counts: {'idle': 39616, 'active': 23690}
Test  mode counts: {'active': 109959, 'idle': 37778}


In [47]:
# sequence builder
SEQ_LEN=25; 
STRIDE=125; MIN_CONN_PER_GROUP=20

def build_sequences(X, y, g):
    X_seq, y_seq, g_seq = [], [], []
    for grp in np.unique(g):
        idx = np.where(g == grp)[0]
        if len(idx) < MIN_CONN_PER_GROUP:
            continue
        # for 13-way, label is the group itself (same for all in grp)
        yg = int(pd.Series(y[idx]).mode().iloc[0])
        for s in range(0, len(idx), STRIDE):
            win = idx[s:s+SEQ_LEN]
            if len(win) == 0:
                continue
            if len(win) < SEQ_LEN:
                pad = np.zeros((SEQ_LEN-len(win), X.shape[1]), dtype=np.float32)
                seq = np.vstack([X[win], pad])
            else:
                seq = X[win]
            X_seq.append(seq); y_seq.append(yg); g_seq.append(int(grp))
    return np.stack(X_seq), np.array(y_seq), np.array(g_seq)

Xseq_tr, yseq_tr, _ = build_sequences(X_tr, y_tr, g_tr)

In [48]:
classes = np.unique(yseq_tr)
cw = compute_class_weight(class_weight="balanced", classes=classes, y=yseq_tr)
class_weight = {int(c): float(w) for c, w in zip(classes, cw)}

# model
n_feat = Xseq_tr.shape[2]
inp = layers.Input(shape=(SEQ_LEN, n_feat))
x = layers.Masking(mask_value=0.0)(inp)
x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.35)(x)
out = layers.Dense(n_cls, activation="softmax")(x)
model = Model(inp, out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'global_max_pooling1d_8' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [49]:
cb = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]
model.fit(Xseq_tr, yseq_tr, validation_split=0.1, epochs=25, batch_size=64,
          callbacks=cb, class_weight=class_weight, verbose=2)

def eval_mode(mode_name):
    if mode_name == "mix":
        mask = np.ones(len(df_test), dtype=bool)
    else:
        mask = (mode_te == mode_name)

    X_t = X_te[mask]; y_t = y_te[mask]; g_t = g_te[mask]
    if len(X_t) == 0:
        print(mode_name, "empty"); return np.nan

    Xseq_t, yseq_t, gseq_t = build_sequences(X_t, y_t, g_t)
    proba = model.predict(Xseq_t, batch_size=256, verbose=0)

    # safe macro across classes present in this test slice
    classes_in_test = np.unique(yseq_t)
    proba_sub = proba[:, classes_in_test]
    Y_sub = np.eye(n_cls)[yseq_t][:, classes_in_test]
    auprc = average_precision_score(Y_sub, proba_sub, average="macro")

    print(f"[13-way Test:{mode_name}] AUCPR={auprc:.4f} | groups={sorted(set(gseq_t))} | classes={len(classes_in_test)}")
    return auprc

au_idle   = eval_mode("idle")
au_active = eval_mode("active")
au_mix    = eval_mode("mix")

print("\n=== 13-way (label=Group), Train: Mix (time-split within each group) ===")
print("Test: Idle  =", au_idle)
print("Test: Active=", au_active)
print("Test: Mix   =", au_mix)

Epoch 1/25
8/8 - 6s - 789ms/step - loss: 2.2519 - val_loss: 2.8529 - learning_rate: 1.0000e-03
Epoch 2/25
8/8 - 1s - 101ms/step - loss: 2.1150 - val_loss: 3.1811 - learning_rate: 1.0000e-03
Epoch 3/25
8/8 - 1s - 96ms/step - loss: 1.8991 - val_loss: 4.1938 - learning_rate: 1.0000e-03
Epoch 4/25
8/8 - 1s - 158ms/step - loss: 1.7029 - val_loss: 5.0055 - learning_rate: 5.0000e-04


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'global_max_pooling1d_8' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


[13-way Test:idle] AUCPR=0.2680 | groups=[np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)] | classes=8
[13-way Test:active] AUCPR=0.2842 | groups=[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(7), np.int64(12), np.int64(13)] | classes=8
[13-way Test:mix] AUCPR=0.1749 | groups=[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13)] | classes=13

=== 13-way (label=Group), Train: Mix (time-split within each group) ===
Test: Idle  = 0.2680216085169509
Test: Active= 0.28421284552808124
Test: Mix   = 0.17486332737978466
